# Model and embedding details

Use this notebook as an architecture reference during the demo. It explains the embedding dimensions, Transformer settings, and output layer used in the compact Colab demo.

The demo is intentionally small and CPU-friendly. The goal is to prove the multimodal pipeline and representation flow, not to train a production-scale model.

## Demo-level configuration

| Component | Demo value | Meaning |
|---|---:|---|
| Rolling window | `20` | each tabular sample uses 20 historical rows |
| Tabular feature count | `11` | engineered OHLCV / relative-strength features |
| Text embedding dimension | `16` | deterministic text token vector used by the demo |
| Image embedding dimension | `16` | output of the demo image Transformer encoder |
| KG embedding dimension | usually `4` | peer count, peer return, sector return, high-volume flag; exact shape is printed from the NPZ |
| Fusion hidden dimension | `16` | common Transformer dimension for all modalities |
| Fusion attention heads | `4` | multi-head self-attention heads |
| Fusion layers | `1` | one Transformer encoder layer in the demo run |
| Fusion feed-forward dimension | `32` | inner MLP dimension in the Transformer block |

The all-in-one notebook prints actual tensor shapes after building `real_world_multimodal_samples.npz`. Treat those printed shapes as the source of truth for a specific run.

## Architectural note: encoder-only Transformer

This project uses an **encoder-style Transformer**, not an encoder-decoder Transformer.

The current task is binary prediction/classification:

> Given tabular, image, text, and KG signals for a stock/date sample, predict whether the stock will outperform the Nifty benchmark over the configured horizon.

Because the model emits one prediction rather than generating a sequence, a Transformer decoder is not needed.

The architecture is:

```text
modality tokens
  -> projection to shared dimension
  -> TransformerEncoder self-attention
  -> pooled fused representation
  -> dropout
  -> Linear(model_dim -> 1)
  -> binary logit
  -> sigmoid(logit) at inference
```

A decoder would make sense in a future extension if the system needed to generate text, for example an analyst-style explanation or natural-language rationale. That would look like:

```text
multimodal encoder
  -> text decoder / LLM decoder
  -> generated explanation
```

That is not part of the current demo model.

## Modality-by-modality details

### 1. Tabular modality

**Raw input:** OHLCV and benchmark data from yfinance.

**Feature set in the demo:**

```text
log_return_1d
cum_return_3d
cum_return_5d
cum_return_10d
realized_vol_5d
realized_vol_10d
high_low_range_over_close
close_over_10dma_minus_1
close_over_20dma_minus_1
volume_over_20d_avg
stock_minus_index_return
```

**Demo tensor shape:**

```text
tabular_tokens: [num_samples, 20, 11]
```

Each 11-dimensional time-step feature vector is projected into the common fusion dimension of 16. The 20 projected tabular time-step tokens attend with the other modality tokens inside the fusion Transformer.

The repo also contains a standalone `TabularTransformer` baseline. Its default config is model dimension 64, 4 heads, 2 Transformer layers, feed-forward dimension 128, and mean pooling. The all-in-one fusion demo does not use this standalone model directly; it uses the fusion model projection path.

### 2. Image modality

**Raw input:** generated candlestick chart PNGs for the same stock/date sample.

**Demo image encoder:** `ImageTransformer`.

| Parameter | Value |
|---|---:|
| image size | `64 x 64` |
| patch size | `16 x 16` |
| patches | `16` patches plus one CLS token |
| model dimension | `16` |
| attention heads | `4` |
| Transformer encoder layers | `1` |
| feed-forward dimension | `32` |

**Demo output shape:**

```text
image_tokens: [num_samples, 16]
```

The image encoder returns one pooled CLS embedding per chart. That vector is then projected into the fusion model's common 16-dimensional space.

### 3. Text modality

**Raw input:** leakage-safe market-summary text records generated from real OHLCV-derived features.

**Cutoff rule:**

```text
event_date <= sample end_date
```

**Demo encoder:** deterministic lightweight text tokenization via stable hashing. This is CPU-friendly and avoids depending on a large pretrained language model in the demo.

**Demo output shape:**

```text
text_tokens: [num_samples, 16]
```

**Transformer layers for text before fusion:** none in the current demo. Text is converted to a 16-dimensional vector and then projected into the fusion Transformer. A future enhancement could replace this with FinBERT or a compact sentence embedding model.

### 4. Knowledge graph modality

**Raw input:** lightweight graph/context records: stock, sector, peer relationship, recent returns, and high-volume events.

**Demo KG token fields:**

```text
peer_count
peer_avg_recent_return
sector_avg_recent_return
event flag(s), e.g. high_volume
```

**Demo output shape:** usually:

```text
kg_tokens: [num_samples, 4]
```

The exact KG width can change if event types change. The all-in-one notebook prints the actual `kg_tokens` shape after saving the NPZ.

**Transformer layers for KG before fusion:** none in the current demo. KG context is a compact numeric feature vector, then projected into the fusion Transformer. A future enhancement could add graph embeddings or a graph neural network.

## Fusion Transformer architecture

The fusion model receives available modality tokens and projects all of them to a shared hidden size.

| Parameter | Value |
|---|---:|
| common model dimension | `16` |
| attention heads | `4` |
| Transformer encoder layers | `1` |
| feed-forward dimension | `32` |
| activation | `GELU` |
| normalization style | `norm_first=True` |
| pooling | `CLS` |

**Fusion sequence concept:**

```text
[CLS]
+ projected tabular tokens, shape [20, 16]
+ projected image token, shape [1, 16]
+ projected text token, shape [1, 16]
+ projected KG token, shape [1, 16]
-> TransformerEncoder self-attention
-> pooled fused representation
-> binary logit: outperformance vs Nifty
```

For the default all-modality demo, the combined sequence is approximately 24 tokens including CLS.

## Output layer and nonlinearity

The output head is intentionally simple:

```text
pooled fused representation
  -> Dropout
  -> Linear(model_dim -> 1)
  -> binary logit
```

During training, the model should use the raw logit with:

```text
BCEWithLogitsLoss
```

That loss applies the sigmoid internally in a numerically stable way.

During inference, convert the logit into probability with:

```text
probability = sigmoid(logit)
```

So the final nonlinearity is effectively sigmoid, but it is not manually applied before `BCEWithLogitsLoss` during training.

In [ ]:
# Compact architecture summary for the demo.
demo_architecture = {
    'tabular': {
        'shape': '[num_samples, 20, 11]',
        'pre_fusion_transformer_layers': 0,
        'fusion_projection_dim': 16,
    },
    'image': {
        'image_size': 64,
        'patch_size': 16,
        'patch_tokens': 16,
        'encoder_model_dim': 16,
        'encoder_heads': 4,
        'encoder_layers': 1,
        'encoder_ff_dim': 32,
    },
    'text': {
        'shape': '[num_samples, 16]',
        'encoder': 'deterministic stable text token, no text Transformer in demo',
        'fusion_projection_dim': 16,
    },
    'kg': {
        'shape': 'usually [num_samples, 4], inspect NPZ for exact width',
        'encoder': 'compact numeric KG context token, no GNN in demo',
        'fusion_projection_dim': 16,
    },
    'fusion_transformer': {
        'type': 'encoder-only Transformer',
        'decoder': 'not used',
        'model_dim': 16,
        'num_heads': 4,
        'num_layers': 1,
        'ff_dim': 32,
        'activation': 'GELU',
        'pooling': 'CLS',
    },
    'output_head': {
        'layers': 'Dropout -> Linear(model_dim -> 1)',
        'training_loss': 'BCEWithLogitsLoss(raw_logit, label)',
        'inference_probability': 'sigmoid(logit)',
    },
}
demo_architecture

## Suggested narration

> The model is encoder-only. We are not generating text or sequences, so we do not need a Transformer decoder. We use self-attention in the encoder to fuse tabular, image, text, and KG signals, then pool the fused representation and pass it through dropout and a linear output head to produce a binary logit. During training, `BCEWithLogitsLoss` consumes the raw logit and applies sigmoid internally. During inference, sigmoid converts the logit into the probability of outperformance versus the Nifty.